# 03 — Full benchmark sweep

Drive `InferenceEngine` across the sweep grid (batch × seq), profile each point,
compute MFU, and write `benchmark_<backend>_<dtype>.{csv,json}` to `results/`.

**GPU required.** Run the eager/onnx part in the main env (`gpu.txt`); run the
vllm part in a SEPARATE session (`gpu-serve.txt`) — see the vLLM note below.

In [ ]:
# ── Environment check ──────────────────────────────────────────
import sys
sys.path.insert(0, "/content/llm-inference-optimizer")  # Colab path
sys.path.insert(0, "..")                                  # local path fallback

from src.utils.env import get_env_info
info = get_env_info()
print(f"Device   : {info['device']}")
print(f"GPU      : {info['gpu_name']}")
print(f"CUDA     : {info['cuda_version']}")
print(f"PyTorch  : {info['torch_version']}")
print(f"Colab    : {info['is_colab']}")

## 1. Pick the sweep config + model

In [ ]:
from src.utils.config import load_model_config

# Benchmark target tier:
#   "smoke" — tiny-random toy: fast sanity check, throughput is meaningless
#   "t4"    — TinyLlama-1.1B real model on a free T4 (the headline run)  <-- default
#   "a100"  — real LLaMA 3 8B (needs an A100 + HF_TOKEN)
TIER = "t4"

if TIER == "a100":
    MODEL  = load_model_config("llama3_8b")["model_id"]
    CONFIG = "llama3_8b_sweep"
    import os
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"])
elif TIER == "t4":
    MODEL  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    CONFIG = "default_sweep"
else:  # smoke
    MODEL  = "hf-internal-testing/tiny-random-LlamaForCausalLM"
    CONFIG = "default_sweep"

OUTPUT = "results/benchmarks"
print("tier:", TIER, "| model:", MODEL, "| config:", CONFIG)

## 2. Sweep eager + onnx (main env)

`run_sweep` takes ONE engine (fixes backend/dtype), so loop backends and call it
once each. onnx needs an exported ONNX dir as its `model_path`.

In [ ]:
from src.serving.inference_engine import InferenceEngine
from src.benchmarking.benchmark_runner import run_sweep

all_results = []

# eager
eng = InferenceEngine(MODEL, backend="eager", device="cuda")
all_results += run_sweep(CONFIG, eng, OUTPUT)

# onnx — skip for the 8B tier (export too large); runs for smoke/t4
if TIER != "a100":
    import os
    from src.export.onnx_exporter import export_to_onnx
    onnx_dir = export_to_onnx(MODEL, "results/onnx/bench", dtype="fp16", verify_export=False)
    eng = InferenceEngine(os.path.dirname(onnx_dir), backend="onnx", device="cuda")
    all_results += run_sweep(CONFIG, eng, OUTPUT)

print(f"{len(all_results)} results so far")

## 3. Sweep vllm (SEPARATE session)

vLLM needs its own env. In a fresh runtime: install via
`uv pip install vllm --torch-backend=auto`, then run only this cell. The backend
self-heals Colab's CUDA runtime (see `_prepare_vllm_runtime`). Use a real model.

In [ ]:
# Run in a FRESH runtime with vLLM installed (uv pip install vllm --torch-backend=auto).
# Self-contained (separate session = no vars from above).
from src.serving.inference_engine import InferenceEngine
from src.benchmarking.benchmark_runner import run_sweep
from src.utils.config import load_model_config

TIER = "t4"   # match what you ran above: "t4" (TinyLlama) | "a100" (8B) | "smoke"

if TIER == "a100":
    VLLM_MODEL, VLLM_CONFIG = load_model_config("llama3_8b")["model_id"], "llama3_8b_sweep"
    import os
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"])
else:
    VLLM_MODEL, VLLM_CONFIG = "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "default_sweep"

eng = InferenceEngine(VLLM_MODEL, backend="vllm", device="cuda")
vllm_results = run_sweep(VLLM_CONFIG, eng, "results/benchmarks")
print(f"{len(vllm_results)} vllm results")

## 4. Peek at the aggregated results

In [ ]:
import glob, json
for f in sorted(glob.glob("results/benchmarks/*.json")):
    rows = json.load(open(f))
    print(f"\n{f}  ({len(rows)} rows)")
    for r in rows[:3]:
        print(f"  bs={r['batch_size']} seq={r['seq_len']} "
              f"tps={r['throughput_tps']:.1f} mfu={r['mfu_percent']:.1f}% "
              f"pow={r['power_watts']:.1f}W")

In [ ]:
# ── Save results to GitHub ─────────────────────────────────────
import subprocess
NOTEBOOK_NAME = "03_benchmark"
subprocess.run(["git", "add", "results/", "notebooks/"], check=True)
subprocess.run(["git", "commit", "-m", f"results: {NOTEBOOK_NAME} run"], check=True)
subprocess.run(["git", "push"], check=True)
print("Results pushed to GitHub.")